# Ingest cleaned and transformed data into Silver Layer

## Brands

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType

catalog_name = 'ecommerce'

df_bronze = spark.table(f'{catalog_name}.bronze.brz_brands')
df_bronze.show(10)

In [0]:
df_silver = df_bronze.withColumn("brand_name", F.trim(F.col("brand_name")))
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', ''))
df_silver.show(10)

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

# PySpark replace is easy
df_silver = df_silver.replace(to_replace=anomalies, subset=["category_code"])

df_silver.select("category_code").distinct().show()

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

## Categories

In [0]:
categories_bronze=spark.table(f"{catalog_name}.bronze.brz_category")
categories_bronze.show(10)

In [0]:
df_duplicates=categories_bronze.groupBy("category_code").count().filter(F.col("count")>1)
df_duplicates.show()

In [0]:
df_silver=categories_bronze.dropDuplicates(['category_code'])
df_silver.show()

In [0]:
df_silver = df_silver.withColumn("category_code", F.upper(F.col("category_code")))
display(df_silver)

df_silver.write.format('delta').mode('overwrite').option('mergeSchema', 'true').saveAsTable(f"{catalog_name}.silver.slv_category")

## Products

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_products")

row_count, column_count = df_bronze.count(), len(df_bronze.columns)

print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

display(df_bronze.limit(5))

In [0]:
df_bronze.select("weight_grams").show(5, truncate=False)

In [0]:
df_products_silver=df_bronze.withColumn("weight_grams", F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType()))
df_products_silver.select("weight_grams").show(5, truncate=False)

In [0]:
df_products_silver.select("length_cm").show(3)

In [0]:
df_products_silver=df_products_silver.withColumn(
    "length_cm",
    F.regexp_replace(F.col("length_cm"), ",", ".").cast(FloatType())
)
df_products_silver.select("length_cm").show(3)

In [0]:
df_products_silver = df_products_silver.withColumn(
    "category_code",
    F.upper(F.col("category_code"))
).withColumn(
    "brand_code",
    F.upper(F.col("brand_code"))
)
df_products_silver.select("category_code", "brand_code").show(2)

In [0]:
df_products_silver.select("material").distinct().show()

In [0]:
df_products_silver=df_products_silver.withColumn(
    "material",
    F.when(F.col("material")=="Coton", "Cotton")
    .when(F.col("material")=="Ruber", "Rubber")
    .when(F.col("material")=="Alumium", "Aluminium")
    .otherwise(F.col("material"))
)
df_products_silver.select("material").distinct().show()

In [0]:
# Convert negative rating_count to positive
df_products_silver = df_products_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0))  # if null, replace with 0
)

df_products_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10, truncate=False)

# --- Code cell 35 ---
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_dim_products)
df_products_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")

In [0]:
df_bronze1 = spark.read.table(f"{catalog_name}.bronze.brz_customers")

# Get row and column count
row_count, column_count = df_bronze1.count(), len(df_bronze1.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze1.show(10)

In [0]:
null_count = df_bronze1.filter(F.col("customer_id").isNull()).count()
null_count

# There are 300 null values in customer_id column. Display some of those
df_bronze1.filter(F.col("customer_id").isNull()).show(3)

# Drop rows where 'customer_id' is null
df_silver = df_bronze1.dropna(subset=["customer_id"])

# Get row count
row_count = df_silver.count()
print(f"Row count after droping null values: {row_count}")

In [0]:
null_count = df_silver.filter(F.col("phone").isNull()).count()
print(f"Number of nulls in phone: {null_count}")

df_silver.filter(F.col("phone").isNull()).show(3)

In [0]:
### Fill null values with 'Not Available'
df_silver = df_silver.fillna("Not Available", subset=["phone"])

# sanity check (If any nulls still exist)
df_silver.filter(F.col("phone").isNull()).show()

# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_customers)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

In [0]:
df_bronze2=spark.table(f"{catalog_name}.bronze.brz_calendar")
row_count, column_count = df_bronze2.count(), len(df_bronze2.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze2.show(3)

In [0]:
df_bronze2.show(3)

In [0]:
df_bronze2.printSchema()

In [0]:
from pyspark.sql.functions import to_date


# Convert the string column to a date type
df_silver = df_bronze2.withColumn("date", to_date(df_bronze2["date"], "dd-MM-yyyy"))

# --- Code cell 52 ---
print(df_silver.printSchema())

df_silver.show(5)

In [0]:
# Remove Duplicates

# --- Code cell 54 ---
# Find duplicate rows in the DataFrame
duplicates = df_silver.groupBy('date').count().filter("count > 1")

# Show the duplicate rows
print("Total duplicated Rows: ", duplicates.count())
display(duplicates)

# --- Code cell 55 ---
# Remove duplicate rows
df_silver = df_silver.dropDuplicates(['date'])

# Get row count
row_count = df_silver.count()

print("Rows After removing Duplicates: ", row_count)

# --- Markdown cell 56 ---
# `day_name` normalize casing

# --- Code cell 57 ---
# Capitalize first letter of each word in day_name
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))

df_silver.show(5)

# --- Markdown cell 58 ---
# Convert negative `week_of_year` to positive

# --- Code cell 59 ---
df_silver = df_silver.withColumn("week_of_year", F.abs(F.col("week_of_year")))  # Convert negative to positive

df_silver.show(3)

# --- Markdown cell 60 ---
# Enhance `quarter` and `week_of_year` column

# --- Code cell 61 ---
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))

df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))

df_silver.show(3)

# --- Markdown cell 62 ---
# Rename columns

# --- Code cell 63 ---
# Rename a column
df_silver = df_silver.withColumnRenamed("week_of_year", "week")

# --- Code cell 64 ---
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_calendar)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")